# Chapter 05. 방법론 선택 의사결정 트리

## 학습 목표

- 인과추론 방법론 분류 체계를 이해한다
- Causal AI Agent의 의사결정 트리 로직을 분석한다
- 데이터 특성에 따른 방법론 선택 기준을 학습한다
- LLM 기반 vs 규칙 기반 방법론 선택을 비교한다

In [5]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional

# 환경 변수 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. 인과추론 방법론 분류 체계

In [6]:
# ============================================================
# 인과추론 방법론 분류 체계
# ============================================================
# CAIS가 지원하는 9가지 방법론을 3개 범주로 분류한다.

methods_taxonomy = {
    "실험적 (Experimental)": {
        "설명": "무작위배정이 이루어진 실험 데이터",
        "방법론": [
            ("diff_in_means", "Difference in Means", "순수 RCT, 공변량 없음"),
            ("linear_regression", "Linear Regression (OLS)", "RCT + 공변량 조정"),
        ],
        "근거 강도": "최고 (Highest)"
    },
    "준실험적 (Quasi-Experimental)": {
        "설명": "자연 실험이나 정책 변화를 활용",
        "방법론": [
            ("difference_in_differences", "이중차분법 (DiD)", "패널 데이터, 시간 구조"),
            ("instrumental_variable", "도구변수법 (IV)", "유효한 도구변수 필요"),
            ("regression_discontinuity_design", "회귀불연속설계 (RDD)", "연속 배정변수 + 임계값"),
        ],
        "근거 강도": "높음 (High)"
    },
    "관찰적 (Observational)": {
        "설명": "관찰 데이터에서 신중한 식별 전략 사용",
        "방법론": [
            ("propensity_score_matching", "성향점수 매칭 (PSM)", "풍부한 공변량, 좋은 겹침"),
            ("propensity_score_weighting", "성향점수 가중 (PSW)", "풍부한 공변량, 약한 겹침"),
            ("backdoor_adjustment", "백도어 조정", "인과 그래프 기반"),
            ("generalized_propensity_score", "일반화 성향점수 (GPS)", "연속형 처치변수"),
        ],
        "근거 강도": "중간 (Medium)"
    }
}

# 분류 체계 출력
for category, info in methods_taxonomy.items():
    print(f"\n{'='*60}")
    print(f"[{category}]")
    print(f"설명: {info['설명']}")
    print(f"근거 강도: {info['근거 강도']}")
    print(f"{'─'*60}")
    for code, name, condition in info["방법론"]:
        print(f"  {name} ({code})")
        print(f"    조건: {condition}")


[실험적 (Experimental)]
설명: 무작위배정이 이루어진 실험 데이터
근거 강도: 최고 (Highest)
────────────────────────────────────────────────────────────
  Difference in Means (diff_in_means)
    조건: 순수 RCT, 공변량 없음
  Linear Regression (OLS) (linear_regression)
    조건: RCT + 공변량 조정

[준실험적 (Quasi-Experimental)]
설명: 자연 실험이나 정책 변화를 활용
근거 강도: 높음 (High)
────────────────────────────────────────────────────────────
  이중차분법 (DiD) (difference_in_differences)
    조건: 패널 데이터, 시간 구조
  도구변수법 (IV) (instrumental_variable)
    조건: 유효한 도구변수 필요
  회귀불연속설계 (RDD) (regression_discontinuity_design)
    조건: 연속 배정변수 + 임계값

[관찰적 (Observational)]
설명: 관찰 데이터에서 신중한 식별 전략 사용
근거 강도: 중간 (Medium)
────────────────────────────────────────────────────────────
  성향점수 매칭 (PSM) (propensity_score_matching)
    조건: 풍부한 공변량, 좋은 겹침
  성향점수 가중 (PSW) (propensity_score_weighting)
    조건: 풍부한 공변량, 약한 겹침
  백도어 조정 (backdoor_adjustment)
    조건: 인과 그래프 기반
  일반화 성향점수 (GPS) (generalized_propensity_score)
    조건: 연속형 처치변수


## 2. Causal Agent 의사결정 트리 분석

In [3]:
# ============================================================
# 의사결정 트리를 시각적으로 표현한다
# ============================================================

decision_tree_text = """
CAIS 방법론 선택 의사결정 트리
══════════════════════════════

[1] is_rct == True? (무작위 실험인가?)
 ├─ Yes
 │   ├─ instrument_var 존재? → IV (격려 설계, Encouragement Design)
 │   ├─ covariates 존재?    → Linear Regression (공변량 조정 RCT)
 │   └─ 순수 RCT           → Diff-in-Means (단순 평균 차이)
 │
 └─ No (관찰 데이터)
     │
     [2] has_temporal + time_var? (시간 구조가 있는가?)
      ├─ Yes → DiD (이중차분법)
      │
      [3] running_var + cutoff? (배정변수와 임계값이 있는가?)
       ├─ Yes → RDD (회귀불연속설계)
       │
       [4] treatment_type == "binary"? (이진 처치인가?)
        ├─ Yes
        │   ├─ instrument_var?         → IV (도구변수법)
        │   ├─ covariates + 좋은 겹침  → PSM (성향점수 매칭)
        │   ├─ covariates + 약한 겹침  → PSW (성향점수 가중)
        │   └─ fallback               → Linear Regression
        │
        └─ No (연속형/범주형 처치)
            ├─ instrument_var?         → IV
            └─ fallback               → Linear Regression
"""

print(decision_tree_text)


CAIS 방법론 선택 의사결정 트리
══════════════════════════════

[1] is_rct == True? (무작위 실험인가?)
 ├─ Yes
 │   ├─ instrument_var 존재? → IV (격려 설계, Encouragement Design)
 │   ├─ covariates 존재?    → Linear Regression (공변량 조정 RCT)
 │   └─ 순수 RCT           → Diff-in-Means (단순 평균 차이)
 │
 └─ No (관찰 데이터)
     │
     [2] has_temporal + time_var? (시간 구조가 있는가?)
      ├─ Yes → DiD (이중차분법)
      │
      [3] running_var + cutoff? (배정변수와 임계값이 있는가?)
       ├─ Yes → RDD (회귀불연속설계)
       │
       [4] treatment_type == "binary"? (이진 처치인가?)
        ├─ Yes
        │   ├─ instrument_var?         → IV (도구변수법)
        │   ├─ covariates + 좋은 겹침  → PSM (성향점수 매칭)
        │   ├─ covariates + 약한 겹침  → PSW (성향점수 가중)
        │   └─ fallback               → Linear Regression
        │
        └─ No (연속형/범주형 처치)
            ├─ instrument_var?         → IV
            └─ fallback               → Linear Regression



In [7]:
# ============================================================
# 방법론 선택 의사결정 트리 — 자체 구현 (외부 라이브러리 없음)
# ============================================================
# cais 라이브러리 없이, 위 섹션에서 설명한 의사결정 트리를
# 파이썬 함수로 직접 구현한다.

# --- 9가지 방법론의 핵심 가정 사전 ---
METHOD_ASSUMPTIONS = {
    "diff_in_means": [
        "treatment is randomly assigned (or as-if random)",
        "no spillover effects",
        "stable unit treatment value assumption (SUTVA)",
    ],
    "linear_regression": [
        "linearity between covariates and outcome",
        "no omitted variable bias (selection on observables)",
        "homoscedasticity (or use robust SE)",
    ],
    "difference_in_differences": [
        "parallel trends assumption",
        "no anticipation effects",
        "stable composition of groups over time",
    ],
    "instrumental_variable": [
        "instrument relevance (strong first stage)",
        "exclusion restriction (instrument affects outcome only through treatment)",
        "monotonicity (no defiers)",
    ],
    "regression_discontinuity_design": [
        "continuity of potential outcomes at cutoff",
        "no manipulation of the running variable",
        "local randomization near the cutoff",
    ],
    "propensity_score_matching": [
        "conditional independence (selection on observables)",
        "common support / overlap",
        "no unmeasured confounders",
        "correct specification of propensity score model",
    ],
    "propensity_score_weighting": [
        "conditional independence (selection on observables)",
        "positivity (all units have nonzero probability of treatment)",
        "no unmeasured confounders",
        "correct specification of propensity score model",
    ],
    "backdoor_adjustment": [
        "all backdoor paths are blocked by observed covariates",
        "no unmeasured confounders on any backdoor path",
        "correct causal graph specification",
    ],
    "generalized_propensity_score": [
        "weak unconfoundedness given covariates",
        "correct specification of the GPS model",
        "common support across treatment levels",
    ],
}


def select_method(info: dict) -> dict:
    """
    데이터 특성 딕셔너리를 입력받아 최적의 인과추론 방법론을 선택한다.

    Parameters
    ----------
    info : dict
        treatment_variable, outcome_variable, is_rct, covariates,
        instrument_variable, has_temporal_structure, time_variable,
        running_variable, cutoff_value, treatment_variable_type 등

    Returns
    -------
    dict with keys: selected_method, method_justification, method_assumptions
    """
    is_rct = info.get("is_rct", False)
    covariates = info.get("covariates", [])
    instrument = info.get("instrument_variable")
    has_temporal = info.get("has_temporal_structure", False)
    time_var = info.get("time_variable")
    running_var = info.get("running_variable")
    cutoff = info.get("cutoff_value")
    treat_type = info.get("treatment_variable_type", "binary")

    # --- 의사결정 트리 ---
    if is_rct:
        # RCT + 도구변수 → 격려 설계(IV)
        if instrument:
            method = "instrumental_variable"
            reason = (
                f"RCT with instrument '{instrument}'—"
                "use IV for encouragement design."
            )
        elif covariates:
            method = "linear_regression"
            reason = "RCT with covariates—use OLS for precision."
        else:
            method = "diff_in_means"
            reason = "Pure RCT without covariates—difference-in-means."
    else:
        # 관찰 데이터: 시간 구조 → DiD
        if has_temporal and time_var:
            method = "difference_in_differences"
            reason = (
                f"Temporal structure via '{time_var}'—"
                "consider Difference-in-Differences (assumes parallel trends)."
            )
        # 배정변수 + 임계값 → RDD
        elif running_var and cutoff is not None:
            method = "regression_discontinuity_design"
            reason = (
                f"Running variable '{running_var}' with cutoff {cutoff}—"
                "consider RDD."
            )
        # 이진 처치
        elif treat_type == "binary":
            if instrument:
                method = "instrumental_variable"
                reason = (
                    f"Instrument '{instrument}' candidate for binary treatment."
                )
            elif covariates:
                method = "propensity_score_matching"
                reason = "Observational binary treatment with covariates—PSM."
            else:
                method = "linear_regression"
                reason = "Binary treatment, no special structure—fallback to OLS."
        # 연속형/범주형 처치
        else:
            if instrument:
                method = "instrumental_variable"
                reason = (
                    f"Instrument '{instrument}' candidate for "
                    "non-binary treatment."
                )
            else:
                method = "linear_regression"
                reason = "Non-binary treatment, no instrument—fallback to OLS."

    return {
        "selected_method": method,
        "method_justification": reason,
        "method_assumptions": METHOD_ASSUMPTIONS.get(method, []),
    }


# ============================================================
# 5가지 시나리오로 의사결정 트리를 테스트한다
# ============================================================

# 테스트 1: 순수 RCT 데이터
print("=== 테스트 1: 순수 RCT (hospital_treatment.csv) ===")
result1 = select_method({
    "treatment_variable": "treatment",
    "outcome_variable": "recovery",
    "is_rct": True,
    "covariates": [],
    "treatment_variable_type": "binary",
})
print(f"선택: {result1['selected_method']}")
print(f"근거: {result1['method_justification']}")
print(f"가정: {result1['method_assumptions']}")
print()

# 테스트 2: RCT + 공변량
print("=== 테스트 2: RCT + 공변량 (online_classroom.csv) ===")
result2 = select_method({
    "treatment_variable": "format_ol",
    "outcome_variable": "falsexam",
    "is_rct": True,
    "covariates": ["gender", "age"],
    "treatment_variable_type": "binary",
})
print(f"선택: {result2['selected_method']}")
print(f"근거: {result2['method_justification']}")
print()

# 테스트 3: IV 데이터
print("=== 테스트 3: 도구변수 (wage.csv) ===")
result3 = select_method({
    "treatment_variable": "educ",
    "outcome_variable": "wage",
    "is_rct": False,
    "instrument_variable": "quarter_of_birth",
    "covariates": ["age", "race"],
    "treatment_variable_type": "continuous",
})
print(f"선택: {result3['selected_method']}")
print(f"근거: {result3['method_justification']}")
print()

# 테스트 4: 시간 구조 (DiD)
print("=== 테스트 4: 시간 구조 (billboard_impact.csv) ===")
result4 = select_method({
    "treatment_variable": "billboard",
    "outcome_variable": "sales",
    "is_rct": False,
    "has_temporal_structure": True,
    "time_variable": "month",
    "covariates": ["region"],
    "treatment_variable_type": "binary",
})
print(f"선택: {result4['selected_method']}")
print(f"근거: {result4['method_justification']}")
print()

# 테스트 5: RDD 데이터
print("=== 테스트 5: 회귀불연속 (drinking.csv) ===")
result5 = select_method({
    "treatment_variable": "legal_drinking",
    "outcome_variable": "accidents",
    "is_rct": False,
    "running_variable": "age",
    "cutoff_value": 21,
    "covariates": [],
    "treatment_variable_type": "binary",
})
print(f"선택: {result5['selected_method']}")
print(f"근거: {result5['method_justification']}")

=== 테스트 1: 순수 RCT (hospital_treatment.csv) ===
선택: diff_in_means
근거: Pure RCT without covariates—difference-in-means.
가정: ['treatment is randomly assigned (or as-if random)', 'no spillover effects', 'stable unit treatment value assumption (SUTVA)']

=== 테스트 2: RCT + 공변량 (online_classroom.csv) ===
선택: linear_regression
근거: RCT with covariates—use OLS for precision.

=== 테스트 3: 도구변수 (wage.csv) ===
선택: instrumental_variable
근거: Instrument 'quarter_of_birth' candidate for non-binary treatment.

=== 테스트 4: 시간 구조 (billboard_impact.csv) ===
선택: difference_in_differences
근거: Temporal structure via 'month'—consider Difference-in-Differences (assumes parallel trends).

=== 테스트 5: 회귀불연속 (drinking.csv) ===
선택: regression_discontinuity_design
근거: Running variable 'age' with cutoff 21—consider RDD.


## 3. 28개 가정 매핑

In [8]:
# ============================================================
# 각 방법론의 핵심 가정을 표로 정리한다
# ============================================================
# Cell 6에서 정의한 METHOD_ASSUMPTIONS 딕셔너리를 활용한다

print("=== 방법론별 핵심 가정 ===\n")
for method, assumptions in METHOD_ASSUMPTIONS.items():
    print(f"[{method}]")
    for i, assumption in enumerate(assumptions, 1):
        print(f"  {i}. {assumption}")
    print()

# 총 가정 수
total = sum(len(v) for v in METHOD_ASSUMPTIONS.values())
print(f"총 가정 수: {total}개 (across {len(METHOD_ASSUMPTIONS)} methods)")

=== 방법론별 핵심 가정 ===

[diff_in_means]
  1. treatment is randomly assigned (or as-if random)
  2. no spillover effects
  3. stable unit treatment value assumption (SUTVA)

[linear_regression]
  1. linearity between covariates and outcome
  2. no omitted variable bias (selection on observables)
  3. homoscedasticity (or use robust SE)

[difference_in_differences]
  1. parallel trends assumption
  2. no anticipation effects
  3. stable composition of groups over time

[instrumental_variable]
  1. instrument relevance (strong first stage)
  2. exclusion restriction (instrument affects outcome only through treatment)
  3. monotonicity (no defiers)

[regression_discontinuity_design]
  1. continuity of potential outcomes at cutoff
  2. no manipulation of the running variable
  3. local randomization near the cutoff

[propensity_score_matching]
  1. conditional independence (selection on observables)
  2. common support / overlap
  3. no unmeasured confounders
  4. correct specification of prope

## 4. LLM 기반 방법론 선택

In [9]:
# ============================================================
# LLM 기반 방법론 선택 — OpenAI Structured Outputs 활용
# ============================================================
# CAIS의 규칙 기반 트리를 LLM으로 보완/대체하는 접근이다.

class MethodRecommendation(BaseModel):
    """LLM이 추천하는 인과추론 방법론이다."""
    selected_method: str = Field(description="추천 방법론 코드명")
    method_name_kr: str = Field(description="방법론 한국어 이름")
    confidence: float = Field(description="추천 신뢰도 (0~1)")
    justification: str = Field(description="추천 근거 (2~3문장)")
    key_assumptions: List[str] = Field(description="이 방법론의 핵심 가정 (한국어)")
    alternative_methods: List[str] = Field(description="대안 방법론 코드명 리스트")
    data_requirements: str = Field(description="데이터 요구사항")

# 테스트 시나리오들
scenarios = [
    {
        "query": "최저임금 인상이 고용에 미치는 효과를 분석하고 싶다. 여러 주(state)의 월별 패널 데이터가 있다.",
        "data_info": "컬럼: state, month, min_wage_increased, employment_rate, gdp, population. 패널 데이터, 50개 주, 24개월"
    },
    {
        "query": "장학금 수혜가 대학 졸업률에 미치는 효과를 분석하고 싶다. GPA 기준점(3.0) 이상이면 장학금을 받는다.",
        "data_info": "컬럼: student_id, gpa, scholarship, graduated, income, age. GPA가 연속형, scholarship는 이진형"
    },
    {
        "query": "흡연이 폐암 발생에 미치는 효과를 관찰 데이터로 분석하고 싶다. 나이, 성별, 직업 등 공변량이 있다.",
        "data_info": "컬럼: smoking, lung_cancer, age, gender, occupation, bmi, exercise. 관찰 데이터, n=5000"
    }
]

AVAILABLE_METHODS = """diff_in_means, linear_regression, difference_in_differences, 
instrumental_variable, regression_discontinuity_design, propensity_score_matching, 
propensity_score_weighting, backdoor_adjustment, generalized_propensity_score"""

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{'='*60}")
    print(f"시나리오 {i}: {scenario['query'][:50]}...")
    print(f"{'='*60}")
    
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"""너는 인과추론 방법론 전문가이다.
사용 가능한 방법론: {AVAILABLE_METHODS}
데이터와 질문을 분석하여 최적의 방법론을 추천한다."""},
            {"role": "user", "content": f"질문: {scenario['query']}\n\n데이터 정보: {scenario['data_info']}"}
        ],
        response_format=MethodRecommendation
    )
    
    rec = response.choices[0].message.parsed
    print(f"추천: {rec.selected_method} ({rec.method_name_kr})")
    print(f"신뢰도: {rec.confidence:.0%}")
    print(f"근거: {rec.justification}")
    print(f"핵심 가정: {rec.key_assumptions[:2]}")
    print(f"대안: {rec.alternative_methods}")


시나리오 1: 최저임금 인상이 고용에 미치는 효과를 분석하고 싶다. 여러 주(state)의 월별 패널 데...
추천: difference_in_differences (차이의 차이)
신뢰도: 95%
근거: 최저임금 인상이 고용에 미치는 영향을 분석하기 위해서는 사전 및 사후 기간 동안의 변화를 비교하는 것이 필요하다. 차이의 차이 방법은 이러한 변화를 효과적으로 추정할 수 있게 해준다. 또한, 패널 데이터가 있기 때문에 시간과 개체 간의 변화를 모두 활용할 수 있다.
핵심 가정: ['평행 추세 가정: 최저임금 인상이 없었던 그룹과 있었던 그룹이 시간에 따라 유사한 방식으로 변화해야 한다.', '최저임금 인상이 고용에 미치는 영향 외에는 다른 요인이 결과에 영향을 미치지 않아야 한다.']
대안: ['linear_regression', 'instrumental_variable', 'regression_discontinuity_design']

시나리오 2: 장학금 수혜가 대학 졸업률에 미치는 효과를 분석하고 싶다. GPA 기준점(3.0) 이상이면...
추천: regression_discontinuity_design (회귀 불연속 설계)
신뢰도: 85%
근거: GPA 기준점인 3.0을 중심으로 장학금 수혜자와 비수혜자의 졸업률 차이를 비교하면, 장학금이 졸업률에 미치는 효과를 명확하게 파악할 수 있습니다. 이 방법은 불연속점에서의 비교를 통해 인과관계를 밝혀주는 강력한 도구입니다.
핵심 가정: ['GPA가 3.0을 기준으로 졸업 여부에 미치는 영향이 연속적이지 않고 불연속적이다.', 'GPA와 졸업 여부 간의 관계가 기준점 주변에서 비슷한 형태로 유지된다.']
대안: ['difference_in_differences', 'linear_regression']

시나리오 3: 흡연이 폐암 발생에 미치는 효과를 관찰 데이터로 분석하고 싶다. 나이, 성별, 직업 등 공...
추천: difference_in_differences (차이의 차이 (Difference in Di

## 5. 규칙 기반 vs LLM 기반 비교

In [10]:
# ============================================================
# 규칙 기반 vs LLM 기반 방법론 선택 비교
# ============================================================

comparison = {
    "항목": ["로직", "장점", "단점", "CAIS에서의 위치", "적합한 상황"],
    "규칙 기반 (Decision Tree)": [
        "if-elif-else 조건문 체인",
        "예측 가능, 재현성 100%, 빠른 실행",
        "경계 케이스 처리 어려움, 확장 시 복잡성 증가",
        "cais/components/decision_tree.py",
        "명확한 데이터 특성, 프로덕션 환경"
    ],
    "LLM 기반 (Structured Output)": [
        "자연어 맥락 이해 + JSON 출력",
        "유연한 판단, 맥락 이해, 근거 설명 자동 생성",
        "비결정적, API 비용, 환각 가능",
        "cais/components/decision_tree_llm.py",
        "모호한 케이스, 탐색적 분석"
    ]
}

df_comp = pd.DataFrame(comparison)
print("=== 규칙 기반 vs LLM 기반 방법론 선택 비교 ===\n")
for _, row in df_comp.iterrows():
    print(f"[{row['항목']}]")
    print(f"  규칙: {row['규칙 기반 (Decision Tree)']}")
    print(f"  LLM:  {row['LLM 기반 (Structured Output)']}")
    print()

print("→ 최적 전략: 규칙 기반을 기본으로, LLM을 보조로 사용하는 하이브리드 접근")
print("→ CAIS의 USE_LLM_DECISION_TREE = False (기본값)이 이 전략을 반영한다")

=== 규칙 기반 vs LLM 기반 방법론 선택 비교 ===

[로직]
  규칙: if-elif-else 조건문 체인
  LLM:  자연어 맥락 이해 + JSON 출력

[장점]
  규칙: 예측 가능, 재현성 100%, 빠른 실행
  LLM:  유연한 판단, 맥락 이해, 근거 설명 자동 생성

[단점]
  규칙: 경계 케이스 처리 어려움, 확장 시 복잡성 증가
  LLM:  비결정적, API 비용, 환각 가능

[CAIS에서의 위치]
  규칙: cais/components/decision_tree.py
  LLM:  cais/components/decision_tree_llm.py

[적합한 상황]
  규칙: 명확한 데이터 특성, 프로덕션 환경
  LLM:  모호한 케이스, 탐색적 분석

→ 최적 전략: 규칙 기반을 기본으로, LLM을 보조로 사용하는 하이브리드 접근
→ CAIS의 USE_LLM_DECISION_TREE = False (기본값)이 이 전략을 반영한다
